In [1]:
# SETUP: installs + imports

!pip install captum --quiet

import os
import numpy as np
import pandas as pd
import torch
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import captum
from sklearn.metrics import f1_score
import copy

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("timm:", timm.__version__, "| albumentations:", A.__version__, "| captum:", captum.__version__)

# CONFIG

CLASSES = ["LumA", "LumB", "Her2", "Basal"]
label2idx = {c: i for i, c in enumerate(CLASSES)}
idx2label = {i: c for c, i in label2idx.items()}

IMG_SIZE = 224
BATCH_SIZE = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

OLD_PREFIX = "/kaggle/working/tiles"
NEW_PREFIX = "/kaggle/input/datasets/anishapanja/bc-xai-patches-labeled-v1/tiles"
SPLIT_CSV_PATH = "/kaggle/input/datasets/anishapanja/bc-xai-patches-with-split/patches_with_split.csv"


# LOAD CSV, FIX PATHS, ENCODE LABELS

df = pd.read_csv(SPLIT_CSV_PATH)
df["patch_path"] = df["patch_path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
df["label_idx"] = df["subtype_clean"].map(label2idx)

print("Total patches:", df.shape[0])
print(df["label_idx"].value_counts())

sample_paths = df["patch_path"].sample(5, random_state=42).tolist()
any_failed = False
for p in sample_paths:
    try:
        Image.open(p).verify()
    except Exception as e:
        print("FAILED:", p, "->", e)
        any_failed = True
if not any_failed:
    print("Sample image paths verified OK.")

print(
    df.drop_duplicates("patient_id")
      .groupby(["subtype_clean", "split"])["patient_id"]
      .count()
      .unstack(fill_value=0)
)


# DATASET CLASS

class PatchDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["patch_path"]).convert("RGB")
        img = np.array(img)
        label = row["label_idx"]
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        return img, label


# TRANSFORMS

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


# SPLITS, DATASETS, SAMPLER, DATALOADERS

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print("train/val/test patch counts:", len(train_df), len(val_df), len(test_df))

train_ds = PatchDataset(train_df, transform=train_transform)
val_ds = PatchDataset(val_df, transform=eval_transform)
test_ds = PatchDataset(test_df, transform=eval_transform)

class_counts = train_df["label_idx"].value_counts().sort_index()
class_weights = 1.0 / class_counts.values
sample_weights = torch.DoubleTensor(train_df["label_idx"].map(lambda x: class_weights[x]).values)

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print("batch shape:", images.shape, labels.shape)
print("class counts in this batch:", torch.bincount(labels, minlength=4))


# MODELS

def build_model(model_name, num_classes=4):
    return timm.create_model(model_name, pretrained=True, num_classes=num_classes)

effnet = build_model("tf_efficientnet_b4")
swin = build_model("swin_tiny_patch4_window7_224")

print("EfficientNet-B4 params:", sum(p.numel() for p in effnet.parameters()))
print("Swin-Tiny params:", sum(p.numel() for p in swin.parameters()))

effnet.eval()
swin.eval()
with torch.no_grad():
    out_eff = effnet(images[:4])
    out_swin = swin(images[:4])
print("EfficientNet-B4 output shape:", out_eff.shape)
print("Swin-Tiny output shape:", out_swin.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

loss_class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=loss_class_weights)

def build_optimizer_and_scheduler(model, num_epochs, lr=1e-4, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    return optimizer, scheduler

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_preds.append(outputs.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    avg_loss = running_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1, all_preds, all_labels

def train_model(model, model_name, train_loader, val_loader, num_epochs, device, lr=1e-4, weight_decay=1e-4):
    model = model.to(device)
    optimizer, scheduler = build_optimizer_and_scheduler(model, num_epochs, lr, weight_decay)
    best_val_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        print(f"[{model_name}] epoch {epoch}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}")
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_macro_f1": val_f1})
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, history


# FULL TRAINING RUN — EfficientNet-B4

NUM_EPOCHS_FULL = 15

effnet_fresh = build_model("tf_efficientnet_b4")

effnet_trained, effnet_history = train_model(
    effnet_fresh,
    "EfficientNet-B4 (full)",
    train_loader,
    val_loader,
    num_epochs=NUM_EPOCHS_FULL,
    device=device,
)

# TEST SET EVALUATION — EfficientNet-B4

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

@torch.no_grad()
def evaluate_full(model, loader, device, classes):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu())
        all_preds.append(probs.argmax(dim=1).cpu())
        all_labels.append(labels)
    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    print(classification_report(all_labels, all_preds, target_names=classes, digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion matrix (rows=true, cols=pred), class order:", classes)
    print(cm)

    auc_per_class = roc_auc_score(all_labels, all_probs, average=None, multi_class="ovr")
    for c, auc in zip(classes, auc_per_class):
        print(f"AUC-ROC ({c} vs rest): {auc:.4f}")

    return all_probs, all_preds, all_labels, cm

print("\n=== EfficientNet-B4 TEST SET RESULTS ===")
eff_test_probs, eff_test_preds, eff_test_labels, eff_cm = evaluate_full(
    effnet_trained, test_loader, device, CLASSES
)


torch.save(effnet_trained.state_dict(), "/kaggle/working/effnet_b4_best.pth")
np.save("/kaggle/working/effnet_test_probs.npy", eff_test_probs)
np.save("/kaggle/working/effnet_test_preds.npy", eff_test_preds)
np.save("/kaggle/working/effnet_test_labels.npy", eff_test_labels)
print("Saved EfficientNet-B4 weights and test predictions to /kaggle/working")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

EfficientNet-B4 params: 17555788
Swin-Tiny params: 27522430
EfficientNet-B4 output shape: torch.Size([4, 4])
Swin-Tiny output shape: torch.Size([4, 4])
Using device: cuda
[EfficientNet-B4 (full)] epoch 1/15 | train_loss=1.0875 | val_loss=2.0853 | val_macro_f1=0.1970
[EfficientNet-B4 (full)] epoch 2/15 | train_loss=0.7066 | val_loss=2.0970 | val_macro_f1=0.2228
[EfficientNet-B4 (full)] epoch 3/15 | train_loss=0.5405 | val_loss=2.2443 | val_macro_f1=0.2383
[EfficientNet-B4 (full)] epoch 4/15 | train_loss=0.4231 | val_loss=2.3291 | val_macro_f1=0.2303
[EfficientNet-B4 (full)] epoch 5/15 | train_loss=0.3452 | val_loss=2.4202 | val_macro_f1=0.2519
[EfficientNet-B4 (full)] epoch 6/15 | train_loss=0.2799 | val_loss=2.2928 | val_macro_f1=0.2811
[EfficientNet-B4 (full)] epoch 7/15 | train_loss=0.2307 | val_loss=2.4525 | val_macro_f1=0.2829
[EfficientNet-B4 (full)] epoch 8/15 | train_loss=0.1926 | val_loss=2.6197 | val_macro_f1=0.2810
[EfficientNet-B4 (full)] epoch 9/15 | train_loss=0.1646 | val